In [0]:
%sql
select * from bronze.metadata.bronze_config 

In [0]:
%sql
select * from bronze.metadata.bronze_audit_log

In [0]:
%sql
select * from bronze.metadata.bronze_audit_log

In [0]:
%sql
DROP TABLE IF EXISTS bronze.metadata.bronze_audit_log;

CREATE TABLE bronze.metadata.bronze_audit_log
(
  run_id STRING,
  source_name STRING,
  dataset_name STRING,
  target_table STRING,
  status STRING,
  event_time TIMESTAMP,
  error_message STRING
);

In [0]:
spark.sql(f"""
    INSERT INTO bronze.metadata.bronze_audit_log
    VALUES
    (
      '{run_id}',
      '{source_name}',
      '{dataset_name}',
      '{target_table}',
      'STARTED',
      current_timestamp(),
      NULL
    )
""")

In [0]:
spark.sql(f"""
    INSERT INTO bronze.metadata.bronze_audit_log
    VALUES
    (
      '{run_id}',
      '{source_name}',
      '{dataset_name}',
      '{target_table}',
      'SUCCESS',
      current_timestamp(),
      NULL
    )
""")

In [0]:
spark.sql(f"""
    INSERT INTO bronze.metadata.bronze_audit_log
    VALUES
    (
      '{run_id}',
      '{source_name}',
      '{dataset_name}',
      '{target_table}',
      'FAILED',
      current_timestamp(),
      '{error_message}'
    )
""")

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    current_date,
    lit,
    col
)

import uuid
import traceback
import re


# ------------------------------------------------------------
# 1. Notebook Parameters
# ------------------------------------------------------------

dbutils.widgets.text("source_name", "netflix")
dbutils.widgets.text("dataset_name", "netflix_movies")

source_name = dbutils.widgets.get("source_name")
dataset_name = dbutils.widgets.get("dataset_name")
run_id = str(uuid.uuid4())


# ------------------------------------------------------------
# 2. Clean column name function
# ------------------------------------------------------------

def clean_column_name(column_name):
    cleaned = column_name.strip().lower()
    cleaned = re.sub(r"[ ,;{}()\n\t=]+", "_", cleaned)
    cleaned = re.sub(r"_+", "_", cleaned)
    cleaned = cleaned.strip("_")
    return cleaned


try:
    # ------------------------------------------------------------
    # 3. Read metadata config
    # ------------------------------------------------------------

    config_df = spark.sql(f"""
        SELECT *
        FROM bronze.metadata.bronze_config
        WHERE source_name = '{source_name}'
          AND dataset_name = '{dataset_name}'
          AND is_active = true
    """)

    if config_df.count() == 0:
        raise Exception(f"No active config found for {source_name}.{dataset_name}")

    config = config_df.collect()[0]

    source_path = config["source_path"]
    file_pattern = config["file_pattern"]
    target_table = config["target_table"]
    delimiter = config["delimiter"]
    header_flag = config["header_flag"]

    schema_path = f"/Volumes/bronze/metadata/bronze_schemas/{source_name}/{dataset_name}"
    checkpoint_path = f"/Volumes/bronze/metadata/bronze_checkpoints/{source_name}/{dataset_name}"


    # ------------------------------------------------------------
    # 4. Insert audit STARTED - append only
    # ------------------------------------------------------------

    spark.sql(f"""
        INSERT INTO bronze.metadata.bronze_audit_log
        VALUES
        (
          '{run_id}',
          '{source_name}',
          '{dataset_name}',
          '{target_table}',
          'STARTED',
          current_timestamp(),
          NULL
        )
    """)


    # ------------------------------------------------------------
    # 5. Read CSV incrementally using Auto Loader
    # ------------------------------------------------------------

    raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_rescued_data")
        .option("header", header_flag)
        .option("delimiter", delimiter)
        .option("inferColumnTypes", "false")
        .option("pathGlobFilter", file_pattern)
        .load(source_path)
    )


    # ------------------------------------------------------------
    # 6. Clean invalid Delta column names
    # ------------------------------------------------------------

    cleaned_columns = [
        col(column_name).alias(clean_column_name(column_name))
        for column_name in raw_df.columns
        if column_name != "_rescued_data"
    ]

    if "_rescued_data" in raw_df.columns:
        df = raw_df.select(
            *cleaned_columns,
            col("_rescued_data")
        )
    else:
        df = raw_df.select(*cleaned_columns)


    # ------------------------------------------------------------
    # 7. Add Bronze audit columns
    # ------------------------------------------------------------

    df = (
        df
        .withColumn("_source_name", lit(source_name))
        .withColumn("_dataset_name", lit(dataset_name))
        .withColumn("_source_file_path", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_ingestion_date", current_date())
        .withColumn("_run_id", lit(run_id))
    )


    # ------------------------------------------------------------
    # 8. Write to Bronze Delta table
    # ------------------------------------------------------------

    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()


    # ------------------------------------------------------------
    # 9. Insert audit SUCCESS - append only
    # ------------------------------------------------------------

    spark.sql(f"""
        INSERT INTO bronze.metadata.bronze_audit_log
        VALUES
        (
          '{run_id}',
          '{source_name}',
          '{dataset_name}',
          '{target_table}',
          'SUCCESS',
          current_timestamp(),
          NULL
        )
    """)


except Exception as e:
    # ------------------------------------------------------------
    # 10. Insert audit FAILED - append only
    # ------------------------------------------------------------

    error_message = traceback.format_exc().replace("'", "''")

    spark.sql(f"""
        INSERT INTO bronze.metadata.bronze_audit_log
        VALUES
        (
          '{run_id}',
          '{source_name}',
          '{dataset_name}',
          '{target_table}',
          'FAILED',
          current_timestamp(),
          '{error_message}'
        )
    """)

    raise e

In [0]:
%sql
UPDATE bronze.metadata.bronze_config
SET source_path = '/Volumes/bronze/metadata/landing_files/*/'
WHERE source_name = 'netflix';

In [0]:
%sql
UPDATE bronze.metadata.bronze_config
SET file_pattern = 'Netflix TV Shows and Movies.csv'
WHERE dataset_name = 'netflix_tv_shows_movies';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'Netflix_stock_history.csv'
WHERE dataset_name = 'netflix_stock_history';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_movies.csv'
WHERE dataset_name = 'netflix_movies';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_reviews.csv'
WHERE dataset_name = 'netflix_reviews';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_titles.csv'
WHERE dataset_name = 'netflix_titles';

UPDATE bronze.metadata.bronze_config
SET file_pattern = 'netflix_tv_shows_detailed_up_to_2025 .csv'
WHERE dataset_name = 'netflix_tv_shows_detailed';

In [0]:
%sql
SELECT source_name, dataset_name, source_path, file_pattern, target_table
FROM bronze.metadata.bronze_config
WHERE source_name = 'netflix'
ORDER BY dataset_name; 

In [0]:
' /Volumes/bronze/metadata/landing_files/*/'